In [0]:
from src.config import (
    CATALOG,
    SCHEMA,
    RAW_TABLE,
    TRAIN_TABLE,
    TEST_TABLE,
    FEATURE_TABLE,
    PREDICTION_TABLE,
    MONITORING_TABLE,
    REGISTERED_MODEL,
    EXPERIMENT_NAME,
    TARGET_COLUMN,
    RANDOM_STATE,
    TEST_SIZE,
)

print("==========================================")
print("CONFIGURATION")
print("==========================================")

print("CATALOG          :", CATALOG)
print("SCHEMA           :", SCHEMA)

print("RAW TABLE        :", RAW_TABLE)
print("TRAIN TABLE      :", TRAIN_TABLE)
print("TEST TABLE       :", TEST_TABLE)
print("FEATURE TABLE    :", FEATURE_TABLE)
print("PREDICTION TABLE :", PREDICTION_TABLE)
print("MONITORING TABLE :", MONITORING_TABLE)

print("REGISTERED MODEL :", REGISTERED_MODEL)
print("EXPERIMENT       :", EXPERIMENT_NAME)

print("TARGET           :", TARGET_COLUMN)
print("RANDOM STATE     :", RANDOM_STATE)
print("TEST SIZE        :", TEST_SIZE)

print("==========================================")

In [0]:
feature_table_exists = spark.catalog.tableExists(
    FEATURE_TABLE
)

print(
    "Feature table exists:",
    feature_table_exists
)

In [0]:
from src.config import (
    FEATURE_TABLE,
    TEST_TABLE,
    REGISTERED_MODEL,
    EXPERIMENT_NAME,
    TARGET_COLUMN,
    RANDOM_STATE,
)

print("FEATURE TABLE   :", FEATURE_TABLE)
print("TEST TABLE      :", TEST_TABLE)
print("REGISTERED MODEL:", REGISTERED_MODEL)
print("EXPERIMENT      :", EXPERIMENT_NAME)

In [0]:
import mlflow
import mlflow.sklearn

import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from mlflow.models import infer_signature

from src.feature_engineering import (
    get_feature_columns
)

print("Libraries imported successfully.")

In [0]:
mlflow.set_experiment(
    EXPERIMENT_NAME
)

print(
    "MLflow experiment:",
    EXPERIMENT_NAME
)

In [0]:
feature_df = spark.table(
    FEATURE_TABLE
)

print(
    "Feature table:",
    FEATURE_TABLE
)

print(
    "Rows:",
    feature_df.count()
)

display(feature_df)

In [0]:
required_columns = (
    ["record_id"]
    + get_feature_columns()
    + [TARGET_COLUMN]
)

missing_columns = [
    column
    for column in required_columns
    if column not in feature_df.columns
]

print(
    "Required columns:",
    required_columns
)

print(
    "Missing columns:",
    missing_columns
)

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print(
    "Training schema validation: PASSED"
)

In [0]:
feature_columns = get_feature_columns()

print("Model features:")

for column in feature_columns:
    print(
        f" - {column}"
    )

In [0]:
train_pdf = (
    feature_df
    .select(
        *feature_columns,
        TARGET_COLUMN
    )
    .toPandas()
)

print(
    "Training DataFrame shape:",
    train_pdf.shape
)

display(train_pdf.head())

In [0]:
X_train = train_pdf[
    feature_columns
]

y_train = train_pdf[
    TARGET_COLUMN
]

print(
    "X_train shape:",
    X_train.shape
)

print(
    "y_train shape:",
    y_train.shape
)

In [0]:
assert len(X_train) == len(y_train)

assert X_train.isnull().sum().sum() == 0

assert y_train.isnull().sum() == 0

assert set(
    y_train.unique()
).issubset({0, 1, 2})

print(
    "Training data validation: PASSED"
)

In [0]:
model_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "min_samples_split": 2,
    "random_state": RANDOM_STATE,
}

print("Model parameters:")

for key, value in model_params.items():
    print(
        f"{key}: {value}"
    )

In [0]:
model = RandomForestClassifier(
    n_estimators=model_params["n_estimators"],
    max_depth=model_params["max_depth"],
    min_samples_split=model_params["min_samples_split"],
    random_state=model_params["random_state"],
    n_jobs=-1,
)

print(model)

In [0]:
with mlflow.start_run(
    run_name="iris_random_forest"
) as run:

    run_id = run.info.run_id

    print(
        "MLflow Run ID:",
        run_id
    )

    # ----------------------------------------
    # Train model
    # ----------------------------------------

    model.fit(
        X_train,
        y_train
    )

    # ----------------------------------------
    # Predictions
    # ----------------------------------------

    train_predictions = model.predict(
        X_train
    )

    # ----------------------------------------
    # Calculate metrics
    # ----------------------------------------

    train_accuracy = accuracy_score(
        y_train,
        train_predictions
    )

    train_precision = precision_score(
        y_train,
        train_predictions,
        average="weighted",
        zero_division=0,
    )

    train_recall = recall_score(
        y_train,
        train_predictions,
        average="weighted",
        zero_division=0,
    )

    train_f1 = f1_score(
        y_train,
        train_predictions,
        average="weighted",
        zero_division=0,
    )

    # ----------------------------------------
    # Log parameters
    # ----------------------------------------

    mlflow.log_params(
        model_params
    )

    # ----------------------------------------
    # Log metrics
    # ----------------------------------------

    mlflow.log_metrics(
        {
            "train_accuracy": train_accuracy,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "train_f1": train_f1,
        }
    )

    # ----------------------------------------
    # Model signature
    # ----------------------------------------

    signature = infer_signature(
        X_train,
        train_predictions
    )

    # ----------------------------------------
    # Log model
    # ----------------------------------------

    mlflow.sklearn.log_model(
        sk_model=model,
        name="iris_random_forest_model",
        signature=signature,
        input_example=X_train.head(3),
    )

    # ----------------------------------------
    # Tags
    # ----------------------------------------

    mlflow.set_tags(
        {
            "project": "iris-mlops",
            "model_type": "RandomForestClassifier",
            "dataset": "Iris",
            "framework": "scikit-learn",
            "environment": "development",
        }
    )

    print(
        "Training completed successfully."
    )

In [0]:
print("==========================================")
print("TRAINING METRICS")
print("==========================================")

print(
    f"Accuracy : {train_accuracy:.4f}"
)

print(
    f"Precision: {train_precision:.4f}"
)

print(
    f"Recall   : {train_recall:.4f}"
)

print(
    f"F1 Score : {train_f1:.4f}"
)

print("==========================================")

In [0]:
cm = confusion_matrix(
    y_train,
    train_predictions
)

print(
    "Training Confusion Matrix:"
)

print(cm)

In [0]:
print(
    classification_report(
        y_train,
        train_predictions,
        target_names=[
            "setosa",
            "versicolor",
            "virginica",
        ],
    )
)

In [0]:
importance_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": model.feature_importances_,
    }
).sort_values(
    "importance",
    ascending=False
)

display(
    importance_df
)

In [0]:
assert (
    len(importance_df)
    == len(feature_columns)
)

assert abs(
    importance_df["importance"].sum() - 1.0
) < 0.0001

print(
    "Feature importance validation: PASSED"
)

In [0]:
run_info = mlflow.get_run(
    run_id
)

print(
    "Run ID:",
    run_info.info.run_id
)

print(
    "Run status:",
    run_info.info.status
)

print(
    "Artifact URI:",
    run_info.info.artifact_uri
)

In [0]:
print("MLflow Parameters:")

for key, value in run_info.data.params.items():
    print(
        f"{key}: {value}"
    )

In [0]:
print("MLflow Metrics:")

for key, value in run_info.data.metrics.items():
    print(
        f"{key}: {value:.4f}"
    )

In [0]:
model_uri = (
    f"runs:/{run_id}/iris_random_forest_model"
)

print(
    "Model URI:"
)

print(
    model_uri
)

In [0]:
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="iris_random_forest_model"
)

print(
    "Model artifact downloaded to:"
)

print(
    artifact_path
)

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL
)

print(
    "Model registered successfully."
)

print(
    "Model name:",
    registered_model.name
)

print(
    "Model version:",
    registered_model.version
)

In [0]:
model_version = registered_model.version

print(
    "Registered model:",
    REGISTERED_MODEL
)

print(
    "Model version:",
    model_version
)

In [0]:
import time

client = mlflow.MlflowClient()

for attempt in range(30):

    model_info = client.get_model_version(
        name=REGISTERED_MODEL,
        version=model_version,
    )

    status = model_info.status

    print(
        f"Attempt {attempt + 1}: "
        f"{status}"
    )

    if status == "READY":
        break

    time.sleep(2)

assert status == "READY", (
    f"Model registration failed. "
    f"Final status: {status}"
)

print(
    "Model registration status: READY"
)

In [0]:
client.update_registered_model(
    name=REGISTERED_MODEL,
    description=(
        "Iris flower classification model "
        "using Random Forest. "
        "Model lifecycle is managed using "
        "MLflow and Unity Catalog."
    ),
)

print(
    "Registered model description updated."
)

In [0]:
client.update_model_version(
    name=REGISTERED_MODEL,
    version=model_version,
    description=(
        "Random Forest model trained on "
        "mlops_demo.iris.iris_features."
    ),
)

print(
    "Model version description updated."
)

In [0]:
client.set_model_version_tag(
    name=REGISTERED_MODEL,
    version=model_version,
    key="model_type",
    value="RandomForestClassifier",
)

client.set_model_version_tag(
    name=REGISTERED_MODEL,
    version=model_version,
    key="dataset",
    value="Iris",
)

client.set_model_version_tag(
    name=REGISTERED_MODEL,
    version=model_version,
    key="feature_table",
    value=FEATURE_TABLE,
)

client.set_model_version_tag(
    name=REGISTERED_MODEL,
    version=model_version,
    key="framework",
    value="scikit-learn",
)

print(
    "Model version tags added."
)

In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias="Challenger",
    version=model_version,
)

print(
    f"Model version {model_version} "
    f"assigned to Challenger."
)

In [0]:
challenger = (
    client.get_model_version_by_alias(
        REGISTERED_MODEL,
        "Challenger"
    )
)

print("==========================================")
print("CHALLENGER MODEL")
print("==========================================")

print(
    "Model name:",
    challenger.name
)

print(
    "Version:",
    challenger.version
)

print(
    "Alias: Challenger"
)

print("==========================================")

In [0]:
assert run_id is not None

assert model_version is not None

assert (
    challenger.version
    == model_version
)

assert (
    run_info.info.status
    == "FINISHED"
)

print("==========================================")
print("03_TRAIN COMPLETED SUCCESSFULLY")
print("==========================================")

print(
    f"MLflow Run ID      : {run_id}"
)

print(
    f"Registered Model   : {REGISTERED_MODEL}"
)

print(
    f"Model Version      : {model_version}"
)

print(
    f"Challenger Version : {challenger.version}"
)

print(
    f"Train Accuracy     : {train_accuracy:.4f}"
)

print(
    f"Train F1           : {train_f1:.4f}"
)

print("==========================================")